# Grounding DINO tiny — DIMER zero-shot object detection tutorial

[![GitHub](https://img.shields.io/badge/GitHub-181717?style=flat&logo=github&logoColor=white)](https://github.com/kurtvalcorza/grounding-dino-detection-pipeline)
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kurtvalcorza/grounding-dino-detection-pipeline/blob/main/tutorials/grounding_dino_detection_colab.ipynb)
[![Hugging Face](https://img.shields.io/badge/%F0%9F%A4%97%20Hugging%20Face-IDEA--Research%2Fgrounding--dino--tiny-ffcc4d?style=flat)](https://huggingface.co/IDEA-Research/grounding-dino-tiny)
[![Upstream](https://img.shields.io/badge/Upstream-IDEA--Research%2FGroundingDINO-181717?style=flat&logo=github&logoColor=white)](https://github.com/IDEA-Research/GroundingDINO)
[![arXiv](https://img.shields.io/badge/arXiv-2303.05499-b31b1b.svg)](https://arxiv.org/abs/2303.05499)

**Profile:** `TASK-INFERENCE`
**Notebook specification:** DIMER Notebook Specification 1.0
**Capability:** zero-shot (open-vocabulary, text-prompted) object detection using the pinned `IDEA-Research/grounding-dino-tiny` weights

This notebook is the executable reference path for the repository capability. It exercises the repository's public pipeline API rather than reimplementing model inference. At inference the image is resized (shortest edge 800, longest edge 1333) and the caller's phrases are joined into one lower-cased, period-separated text (`"rectangle. red circle."`); a Swin-T image backbone and a BERT text encoder are fused and the decoder proposes boxes whose per-token grounding scores pass a **sigmoid**; the pipeline returns each surviving box in xyxy pixel coordinates of the input image, the phrase it grounded to, and its score. **No adaptation occurs:** no training, fine-tuning, in-context conditioning, or preprocessing fitting happens in this notebook — the upstream checkpoint supplies the weights, processor and tokenizer, and this repository adds packaging, snapshot verification, prompt and image validation with named ceilings, a fixed output contract and the `box_iou` helper. The default sample is a synthetic scene drawn in code; the IoU values reported for it are sanity evidence against the boxes you drew, not a benchmark claim.

**Learning objectives:** bootstrap the repository in a fresh runtime, resolve the immutable upstream model revision, draw a synthetic scene with known object boxes, validate image and prompts against the pipeline's ceilings, run text-prompted detection through the public API with explicit caller-owned thresholds, read sigmoid scores and thresholds correctly, check each detection against the drawn box with `box_iou` as sanity evidence, exercise an optional BYOD path, and export machine-readable detections plus an annotated image and provenance.

**This notebook does not demonstrate:** instance or semantic segmentation (see the sibling SAM 2 pipeline), tracking, OCR, captioning, closed-set detection with a fixed class list, mAP or precision/recall evaluation (which needs a labelled box set), or any training. Prompts are free text, so a phrase the model cannot ground still produces boxes for something — the score, not the label, is your only signal.

## Prerequisites

- **Runtime:** a fresh supported runtime (Google Colab or Jupyter, Python 3.12). The default path runs on CPU and uses CUDA automatically when available; inference is float32 on both. CPU is adequate for one small image: the repository's model card records 6.2 s to load and 4.4 s per `detect` on the 320×240 synthetic scene in the Windows venv (Intel Core Ultra 9 275HX). The pinned `torch==2.14.0` install and the ~689 MB checkpoint are the large downloads of the run.
- **Knowledge:** basic Python and PIL; what a bounding box in xyxy pixel coordinates is; what intersection-over-union measures.
- **Data:** the default sample is a deterministic 320×240 scene drawn in code (grey background, one dark rectangle, one red disc) with two prompts naming them, so nothing is downloaded and no private data is needed. Optional BYOD upload is gated off by default so the sample path can run top-to-bottom without interaction. Expected BYOD input: one image file decodable by Pillow (PNG/JPEG/WebP and similar), any colour mode, sides between 16 and 4096 px, plus your own comma-separated prompt phrases (1–16 phrases, at most 48 characters each). Do not upload confidential or restricted data to a hosted notebook environment unless you are authorized to do so. Uploaded inputs remain in the notebook runtime; this pipeline does not send them to a third-party inference API.
- **External access:** GitHub (repository clone) and the Hugging Face Hub (the package's `stage_missing_files` fetches the pinned checkpoint once, because the Git repository does not vendor the weights). No credentials are required.

## 1. Bootstrap the repository and pinned runtime

When the notebook is opened without a repository checkout, this cell clones the repository. Released notebooks default to `main`; automated candidate validation can set `DIMER_TUTORIAL_REF` to an immutable commit or review branch. The repository is installed as a regular (non-editable) package so it is importable in this same runtime; an editable install would only become importable after a restart. Model-facing dependencies (`torch`, `transformers`, `safetensors`, `numpy`, `pillow`) are pinned exactly in `pyproject.toml`. If installation replaces any package that this runtime has already imported, the cell fails with a restart instruction rather than continuing with mixed versions. Look for a dictionary reporting the repository revision, Python, `torch` and `transformers` versions, and whether CUDA is available.

In [ ]:
import importlib
import importlib.metadata
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = 'https://github.com/kurtvalcorza/grounding-dino-detection-pipeline.git'
REPO_NAME = 'grounding-dino-detection-pipeline'
REPO_REF = os.environ.get('DIMER_TUTORIAL_REF', 'main')
SKIP_INSTALL = os.environ.get('DIMER_NOTEBOOK_CI_PREINSTALLED') == '1'
ROOT = Path.cwd()
if not (ROOT / 'pyproject.toml').exists():
    checkout = ROOT / REPO_NAME
    if not checkout.exists():
        subprocess.run(['git', 'clone', '--filter=blob:none', '-q', REPO_URL, str(checkout)], check=True)
    if REPO_REF != 'main':
        subprocess.run(['git', '-C', str(checkout), 'fetch', '--depth', '1', 'origin', REPO_REF], check=True)
        subprocess.run(['git', '-C', str(checkout), 'checkout', '--detach', 'FETCH_HEAD'], check=True)
    else:
        subprocess.run(['git', '-C', str(checkout), 'checkout', '-q', 'main'], check=True)
        subprocess.run(['git', '-C', str(checkout), 'pull', '--ff-only', '-q', 'origin', 'main'], check=True)
    os.chdir(checkout)
    ROOT = Path.cwd()

if not SKIP_INSTALL:
    # Every distribution that is already imported in this runtime is captured before installation,
    # whatever its name (PIL -> pillow), so a pinned install that replaces any loaded package is
    # detected. Distribution metadata is compared with metadata afterwards: torch.__version__ carries
    # a local build label (for example 2.14.0+cu130) that the distribution version omits.
    def _installed_version(distribution):
        try:
            return importlib.metadata.version(distribution)
        except importlib.metadata.PackageNotFoundError:
            return None
    _module_dists = importlib.metadata.packages_distributions()
    _loaded_dists = sorted({d for m in list(sys.modules) for d in _module_dists.get(m.partition('.')[0], ())})
    loaded = {distribution: _installed_version(distribution) for distribution in _loaded_dists}
    # Non-editable install: an editable (.pth) install is not importable until the
    # interpreter restarts, which a fresh hosted runtime cannot do mid-notebook.
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', str(ROOT)], check=True)
    importlib.invalidate_caches()
    stale = []
    for distribution, before in loaded.items():
        installed = _installed_version(distribution)
        if before is not None and before != installed:
            stale.append(f'{distribution}: loaded={before}, installed={installed}')
    if stale:
        raise RuntimeError('Core dependencies changed while older modules were loaded: ' + '; '.join(stale) + '. Restart the runtime, then rerun from the top.')

REPO_SHA = subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip()
import platform, torch, transformers
print({'repository': str(ROOT), 'repository_revision': REPO_SHA, 'requested_ref': REPO_REF, 'python': platform.python_version(), 'torch': torch.__version__, 'transformers': transformers.__version__, 'cuda': torch.cuda.is_available()})

## 2. Draw the synthetic scene or optional BYOD

The default sample is **synthetic** and carries its own reference boxes: a deterministic 320×240 RGB scene is drawn in code — grey background, a dark filled rectangle at `[40, 60, 140, 180]` and a red filled disc whose bounding box is `[200, 80, 280, 160]` — and the prompts name them (`rectangle`, `red circle`). This is the same scene and prompt pair the repository's smoke run used. The drawn boxes are the reference for the IoU sanity check later; they are not a labelled dataset, so nothing here is a precision/recall measurement. The image digest and the prompt text are printed. BYOD is optional and disabled by default; when enabled, upload one image and set `BYOD_PROMPTS` to the phrases you want grounded (no reference boxes exist for it, so no IoU is computed).

The detection thresholds are **caller-owned request parameters**, not pipeline constants: `box_threshold` keeps a box whose best grounding score reaches it, `text_threshold` keeps the phrase tokens that reach it when the box is labelled. Their package defaults (`BOX_THRESHOLD = 0.4`, `TEXT_THRESHOLD = 0.3`) follow the upstream card's usage example, not a calibration; they are exposed here as form parameters and passed explicitly on every call. Before anything expensive runs, this cell surfaces the pipeline's operational ceilings — `MIN_IMAGE_SIDE` (16 px), `MAX_IMAGE_SIDE` (4096 px), `MAX_PROMPTS` (16), `MAX_PROMPT_CHARS` (48), `MAX_TEXT_TOKENS` (256) — and validates the image sides and prompts with clear messages; the pipeline's own `format_prompts` performs the canonical join. Inside the pipeline the image is converted to RGB and resized by the processor; boxes are mapped back to input pixels, and nothing else is dropped or altered. Look for a dictionary naming the sample kind, image size and digest, the prompt text, the thresholds, and the drawn reference boxes.

In [ ]:
import hashlib
import io

import numpy as np
from PIL import Image, ImageDraw

from grounding_dino_detection_pipeline import MAX_IMAGE_SIDE, MAX_PROMPT_CHARS, MAX_PROMPTS, MAX_TEXT_TOKENS, MIN_IMAGE_SIDE, format_prompts

USE_BYOD = False  # @param {type:"boolean"}
BYOD_PROMPTS = 'cat, remote control'  # @param {type:"string"}
box_threshold = 0.4  # @param {type:"number"}
text_threshold = 0.3  # @param {type:"number"}

print({'ceilings': {'MIN_IMAGE_SIDE': MIN_IMAGE_SIDE, 'MAX_IMAGE_SIDE': MAX_IMAGE_SIDE, 'MAX_PROMPTS': MAX_PROMPTS, 'MAX_PROMPT_CHARS': MAX_PROMPT_CHARS, 'MAX_TEXT_TOKENS': MAX_TEXT_TOKENS}})
for name, value in (('box_threshold', box_threshold), ('text_threshold', text_threshold)):
    if not isinstance(value, (int, float)) or not 0.0 <= float(value) <= 1.0:
        raise ValueError(f'{name} must be a number in [0, 1], got {value!r}; adjust the form value and rerun this cell.')
if USE_BYOD:
    from google.colab import files
    uploaded = files.upload()
    image_name = next(iter(uploaded))
    image = Image.open(io.BytesIO(uploaded[image_name]))
    image.load()
    prompts = [phrase.strip() for phrase in BYOD_PROMPTS.split(',') if phrase.strip()]
    drawn_boxes = None
    sample_kind = 'BYOD'
else:
    # Deterministic synthetic scene: no randomness, so no seed is needed and the digest is stable.
    image = Image.new('RGB', (320, 240), (128, 128, 128))
    draw = ImageDraw.Draw(image)
    drawn_boxes = {'rectangle': [40.0, 60.0, 140.0, 180.0], 'red circle': [200.0, 80.0, 280.0, 160.0]}
    draw.rectangle(drawn_boxes['rectangle'], fill=(30, 30, 30))
    draw.ellipse(drawn_boxes['red circle'], fill=(220, 30, 30))
    prompts = list(drawn_boxes)
    image_name = 'synthetic_scene_320x240.png'
    sample_kind = 'synthetic'

width, height = image.size
if min(width, height) < MIN_IMAGE_SIDE or max(width, height) > MAX_IMAGE_SIDE:
    raise ValueError(f'{image_name}: image {image.size} must have both sides within {MIN_IMAGE_SIDE}..{MAX_IMAGE_SIDE} px; resize it and rerun this cell.')
prompt_text = format_prompts(prompts)  # raises with the offending phrase when a prompt breaks a ceiling
image_sha256 = hashlib.sha256(np.asarray(image.convert('RGB')).tobytes()).hexdigest()
print({'sample_kind': sample_kind, 'name': image_name, 'size': image.size, 'rgb_sha256': image_sha256, 'prompts': prompts, 'prompt_text': prompt_text, 'box_threshold': box_threshold, 'text_threshold': text_threshold, 'drawn_boxes': drawn_boxes})

## 3. Stage, verify and resolve the pinned model

Model acquisition goes through the package, not the notebook. The public API pins the exact upstream revision (`MODEL_ID`/`MODEL_REVISION` are imported from the package, never typed here). The Git repository carries `weights/grounding-dino-tiny/dimer-base-manifest.json`, `config.json`, `preprocessor_config.json` and the BERT tokenizer assets (8 small files) but git-ignores the 689 MB `model.safetensors`, so in a fresh clone `stage_missing_files(WEIGHTS_DIR, allow_download=True)` fetches exactly the manifest entries that are absent, at the pinned revision, into the snapshot directory — it prints the list it fetched (`[]` on a warm runtime) and refuses a manifest whose identity differs from the package pins. `verify_snapshot(WEIGHTS_DIR)` then re-hashes every manifest entry (size and SHA-256) and raises on the first mismatch; its returned dict is printed. Only then does `from_pretrained(weights_dir=WEIGHTS_DIR)` load the verified files with `local_files_only=True` and `trust_remote_code=False` — there is no fallback to a different download. The effective model identity and the device chosen (`cuda:0` when available, else `cpu`) are printed before inference.

In [ ]:
from grounding_dino_detection_pipeline import MODEL_ID, MODEL_KEY, MODEL_REVISION, GroundingDINOPipeline, box_iou, stage_missing_files, verify_snapshot
print({'model_id': MODEL_ID, 'revision': MODEL_REVISION})
WEIGHTS_DIR = ROOT / 'weights' / MODEL_KEY
fetched = stage_missing_files(WEIGHTS_DIR, allow_download=True)
print({'weights_dir': str(WEIGHTS_DIR), 'fetched': fetched})
snapshot = verify_snapshot(WEIGHTS_DIR)
print(snapshot)
pipe = GroundingDINOPipeline.from_pretrained(weights_dir=WEIGHTS_DIR)
print({'device': pipe.device})

## 4. Detect and read the scores correctly

`detect` returns a dict with `detections` — a list of `{box, label, score}` **ordered by descending score**, `box` in xyxy pixel coordinates of the input, `label` the grounded phrase text — plus `prompt_text`, the thresholds used, `width`, `height` and the model identity. Each `score` is a **sigmoid grounding score, not a calibrated probability**: it was never fitted to the frequency with which a box is correct, so 0.9 does not mean "90 % likely". The thresholds you passed are the only decision rule; the pipeline ships them as defaults, not as a calibration, and the caller owns them per deployment — raise `box_threshold` when false boxes cost more than missed ones, lower it for recall, and note that a phrase that grounds nothing well may still surface a box just above the threshold. Inference is deterministic on a fixed device and dtype (no sampling, `torch.inference_mode`); CUDA kernel selection can move scores in the third or fourth decimal place and reorder near-ties.

No detection metric is reported: mean average precision needs a labelled box set with a matching vocabulary, and this repository ships none. The only helper is `box_iou(a, b)` (intersection-over-union of two xyxy boxes), the building block a caller would use to compute mAP on their own labelled data. On the synthetic path it is applied to each detection against the drawn box of the same phrase as a **sanity check** that the geometry round-trips; on BYOD no reference exists and none is computed. As recorded in the model card, the repository's CPU smoke on this same scene at the default thresholds returned `red circle` at score 0.932 and `rectangle` at 0.698 with boxes within about 2 px of the drawn ones; that is one observation, not a calibration point. Look for two detections whose labels match the prompts and whose IoU against the drawn boxes is high; a materially different result on your runtime is a signal to check the install, not a measurement.

In [ ]:
result = pipe.detect(image, prompts, box_threshold=box_threshold, text_threshold=text_threshold)
print({'n_detections': len(result['detections']), 'prompt_text': result['prompt_text'], 'box_threshold': result['box_threshold'], 'text_threshold': result['text_threshold'], 'device': pipe.device})
sanity = []
for rank, det in enumerate(result['detections'], start=1):
    line = f"{rank:>2}. score {det['score']:.4f}  label {det['label']!r}  box {[round(v, 1) for v in det['box']]}"
    if drawn_boxes is not None:
        ious = {phrase: box_iou(det['box'], box) for phrase, box in drawn_boxes.items()}
        best_phrase = max(ious, key=ious.get)
        sanity.append({'rank': rank, 'label': det['label'], 'best_drawn_box': best_phrase, 'iou_vs_best_drawn_box': ious[best_phrase], 'label_matches_box': det['label'] == best_phrase})
        line += f"  iou_vs_drawn[{best_phrase!r}] {ious[best_phrase]:.3f}"
    print(line)
metrics = {}
if drawn_boxes is not None:
    metrics['box_iou_vs_drawn_boxes'] = sanity
    print({'note': 'IoU against boxes you drew yourself on a synthetic scene: sanity evidence, not a detection metric'})
else:
    print('No reference boxes exist for a BYOD image, so box_iou is not computed; inspect the annotated PNG instead.')

## 5. Export outputs and provenance

Machine-readable JSON preserves the full result (score-ordered detections with boxes and labels, prompt text, thresholds), the sanity IoU block when computed, the sample identity, digest and drawn boxes, the repository revision, the model identifier, the immutable model revision, and the runtime identity (Python, `torch`, `transformers`, device). The detections are also written as CSV with explicit `image`, `rank`, `label`, `score`, `x0`, `y0`, `x1`, `y1` columns, and an annotated PNG draws every returned box for visual inspection (a supplement to, not a replacement for, the machine-readable files). No credentials are recorded.

In [ ]:
import csv
import json
os.makedirs('outputs', exist_ok=True)
annotated = image.convert('RGB').copy()
draw = ImageDraw.Draw(annotated)
for det in result['detections']:
    draw.rectangle(det['box'], outline=(0, 255, 0), width=2)
    draw.text((det['box'][0] + 2, det['box'][1] + 2), f"{det['label']} {det['score']:.2f}", fill=(0, 255, 0))
annotated.save('outputs/grounding_dino_detection_annotated.png')
payload = {
    'prediction': result,
    'metrics': metrics,
    'sample': {'kind': sample_kind, 'name': image_name, 'size': list(image.size), 'rgb_sha256': image_sha256, 'prompts': prompts, 'drawn_boxes': drawn_boxes},
    'repository_revision': REPO_SHA,
    'model_id': MODEL_ID,
    'model_revision': MODEL_REVISION,
    'runtime': {
        'python': platform.python_version(),
        'torch': torch.__version__,
        'transformers': transformers.__version__,
        'device': pipe.device,
    },
}
with open('outputs/grounding_dino_detection_result.json', 'w', encoding='utf-8') as handle:
    json.dump(payload, handle, indent=2, ensure_ascii=False)
with open('outputs/grounding_dino_detection_detections.csv', 'w', encoding='utf-8', newline='') as handle:
    writer = csv.writer(handle)
    writer.writerow(['image', 'rank', 'label', 'score', 'x0', 'y0', 'x1', 'y1'])
    for rank, det in enumerate(result['detections'], start=1):
        writer.writerow([image_name, rank, det['label'], f"{det['score']:.6f}", *[f"{v:.2f}" for v in det['box']]])
print(sorted(os.listdir('outputs')))

## Interpretation and limits

The boxes are grounded to free-text phrases: the label tells you which phrase the box scored best against, not that the object is really there, and the sigmoid score is uncalibrated. The thresholds are request parameters you own; the defaults are the upstream usage example, not a tuned operating point. On the synthetic scene the IoU values compare detections to shapes you drew yourself and prove only that the input contract, prompt formatting, forward pass and coordinate mapping work; they say nothing about photographs, small or occluded objects, crowded scenes, or vocabulary the model has never grounded, and a BYOD result is a single-image observation. Long or many prompts are refused at the stated ceilings, and phrases are lower-cased and period-joined before encoding, which can merge or split labels in ways you should inspect in `prompt_text`. The pipeline provides no segmentation, tracking, OCR, captioning, mAP evaluation, or training capability.

Successful execution proves that the recorded repository revision can acquire the pinned model, validate the demonstrated input, execute the public pipeline path, and emit the shown machine-readable outputs in the tested runtime. It does **not** establish benchmark superiority, deployment calibration, safety for high-consequence decisions, or production fitness on an unseen domain.

**Next experiments:** lower `box_threshold` to 0.2 and count how many extra boxes appear on the same scene (the smoke run found none, but a photograph behaves differently); add a phrase that names nothing in the image (`"bicycle"`) and watch where its box lands and how its score compares; enable `USE_BYOD` with a photograph and hand-label a few boxes to compute `box_iou` per object yourself — the first step towards a real precision/recall number.

## References

- Repository README: `../README.md`
- Repository model card: `../MODEL_CARD.md`
- Weight provenance: `../docs/WEIGHTS.md`
- Upstream model: https://huggingface.co/IDEA-Research/grounding-dino-tiny
- Upstream code: https://github.com/IDEA-Research/GroundingDINO
- Grounding DINO: Marrying DINO with Grounded Pre-Training for Open-Set Object Detection (Liu et al., 2023): https://arxiv.org/abs/2303.05499